# 🌡️ **Thermo-physiological Indices**
---

### **Learning Goals:**
- 🎯 **Goal 1 | Physiologically Equivalent Temperature (PET)**: Compute hourly PET values and classify diurnal thermal stress for four days spanning the four seasons of 2012 in Ioannina
- 🎯 **Goal 2 | Universal Thermal Climate Index (UTCI)**: Screen meteorological inputs against operational validity bounds, compute hourly UTCI values, and classify thermal stress for the same four days of 2012 in Ioannina
- 🎯 **Goal 3 | Comparing PET and UTCI**: Quantify numerical and stress-category differences between PET and UTCI and assess sensitivity to outdoor radiative load
- 🎯 **Goal 4 | Gridded UTCI Over Europe**: Compute gridded UTCI over Europe from ECMWF open-data forecasts, map thermal stress categories, and print summary statistics for Greece

**Run the tutorial via Binder:**

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/one-weather-lab/biometeorology-air-quality-and-human-health/HEAD?labpath=notebooks%2F02_thermophysiological_indices.ipynb)

## ⚙️ Setup and Configuration
---
The two cells below import the required packages, set the local data paths and seasonal dates for Ioannina, and define the ECMWF open-data forecast and domain parameters.

In [ ]:
from importlib.metadata import version
from pathlib import Path
import sys

import biometeo
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure the repository root is on sys.path when running from notebooks/
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import (
    PET_SCALE,
    UTCI_SCALE,
    classify,
    mask_utci_inputs,
    plot_europe_utci_map,
    plot_pet_diurnal,
    plot_pet_utci_comparison,
    plot_utci_diurnal,
    read_four_seasonal_days,
)

for pkg in ("biometeo", "numpy", "pandas", "matplotlib"):
    print(f"{pkg:12s} {version(pkg)}")

In [ ]:
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Expected data directory at {DATA_DIR}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEASONAL_DATES = {
    "2012-01-31": "Winter (31 Jan 2012)",
    "2012-04-26": "Spring (26 Apr 2012)",
    "2012-07-11": "Summer (11 Jul 2012)",
    "2012-10-02": "Autumn (2 Oct 2012)",
}

# ECMWF open-data settings for Section 4 (used in the gridded Europe workflow)
RUN_DATE = "20260623"   # None = latest live run; "YYYYMMDD" = retrospective Azure mirror
RUN_TIME = 0            # [UTC hour] forecast cycle time
FORECAST_STEP_H = 12    # [h] valid at 12 UTC
RADIATION_WINDOW_H = 3  # [h] step differencing window (9 h to 12 h)
EUROPE_BOX = (20.0, 75.0, -30.0, 60.0)  # (south, north, west, east) [deg]
STATS_REGIONS = {
    "Greece": (34.5, 42.0, 19.0, 29.5),
    "Europe": (34.0, 72.0, -25.0, 45.0),
}
STATS_REGION = "Greece"

print(f"Data directory   : {DATA_DIR.relative_to(REPO_ROOT)}")
print(f"Output directory : {OUTPUT_DIR.relative_to(REPO_ROOT)}")
print(f"Seasonal days    : {', '.join(SEASONAL_DATES)}")
print(f"Open-data run    : {RUN_DATE or 'live'} {RUN_TIME:02d} UTC (+{FORECAST_STEP_H} h), region = {STATS_REGION}")

## Section 1: Physiologically Equivalent Temperature (PET)

In [ ]:
df_days = read_four_seasonal_days(data_dir=DATA_DIR)

# biometeo.PET signature accepts keyword arguments explicitly so Ta, VP, Tmrt, v cannot cross
df_days["pet"] = [
    float(
        biometeo.PET(
            Ta=row.ta,
            VP=row.vp,
            Tmrt=row.tmrt,
            v=row.ws1_1,
        )["PET_v"]
    )
    for row in df_days.itertuples(index=False)
]

df_days["date_str"] = df_days["time_utc"].dt.strftime("%Y-%m-%d")

def _fmt_minmax(sub: pd.DataFrame, col: str, unit: str) -> str:
    idx_min = sub[col].idxmin()
    idx_max = sub[col].idxmax()
    h_min = sub.loc[idx_min, "time_utc"].hour
    h_max = sub.loc[idx_max, "time_utc"].hour
    return (
        f"{sub.loc[idx_min, col]:6.2f} ({h_min:02d}Z) .. "
        f"{sub.loc[idx_max, col]:6.2f} ({h_max:02d}Z) {unit}"
    )

print(f"Loaded {len(df_days)} hourly records across {df_days['date_str'].nunique()} days in Ioannina.")
print(f"{'Day':24s} {'Ta [C]':27s} {'Tmrt [C]':28s} {'ws1_1 [m/s]':30s} {'PET [C]':27s}")
print("-" * 140)
for date_str, label in SEASONAL_DATES.items():
    sub = df_days[df_days["date_str"] == date_str]
    print(
        f"{label:24s} "
        f"{_fmt_minmax(sub, 'ta', 'C'):27s} "
        f"{_fmt_minmax(sub, 'tmrt', 'C'):28s} "
        f"{_fmt_minmax(sub, 'ws1_1', 'm/s'):30s} "
        f"{_fmt_minmax(sub, 'pet', 'C'):27s}"
    )

In [ ]:
df_days["pet_class"] = classify(values=df_days["pet"], scale=PET_SCALE)

pet_order = [band.label for band in PET_SCALE.bands]
pet_counts = (
    pd.crosstab(df_days["pet_class"], df_days["date_str"])
    .reindex(index=pet_order, columns=list(SEASONAL_DATES), fill_value=0)
)
pet_counts.columns = [SEASONAL_DATES[d] for d in pet_counts.columns]
print("Hourly PET stress-class counts per day (24 h each):")
print(pet_counts.to_string())

pet_fig_path = plot_pet_diurnal(
    df=df_days,
    scale=PET_SCALE,
    output_path=OUTPUT_DIR / "fig_1.2_pet_diurnal.png",
    season_labels=SEASONAL_DATES,
)

## Section 2: Universal Thermal Climate Index (UTCI)

In [ ]:
df_days["utci_valid"] = mask_utci_inputs(
    ta=df_days["ta"],
    tmrt=df_days["tmrt"],
    ws10=df_days["ws10"],
    vp=df_days["vp"],
)

utci_values = np.full(len(df_days), np.nan, dtype=float)
for idx, row in enumerate(df_days.itertuples(index=False)):
    if row.utci_valid:
        utci_values[idx] = float(
            biometeo.UTCI(
                Ta=row.ta,
                VP=row.vp,
                Tmrt=row.tmrt,
                v=row.ws10,
            )
        )
df_days["utci"] = utci_values

n_valid = int(df_days["utci_valid"].sum())
n_total = len(df_days)
n_blank = n_total - n_valid
print(f"UTCI screening: {n_valid} / {n_total} hours within validity bounds; {n_blank} blank hour(s).")

excluded = df_days[~df_days["utci_valid"]]
for row in excluded.itertuples(index=False):
    reasons = []
    if not (-50.0 <= row.ta <= 50.0):
        reasons.append(f"ta = {row.ta:.2f} C outside [-50, +50] C")
    if not (0.5 <= row.ws10 <= 17.0):
        reasons.append(f"ws10 = {row.ws10:.4f} m/s < 0.50 m/s lower bound" if row.ws10 < 0.5 else f"ws10 = {row.ws10:.4f} m/s > 17.0 m/s")
    if not (-30.0 <= (row.tmrt - row.ta) <= 70.0):
        reasons.append(f"tmrt - ta = {row.tmrt - row.ta:.2f} K outside [-30, +70] K")
    if not (0.0 <= row.vp <= 50.0):
        reasons.append(f"vp = {row.vp:.2f} hPa outside [0, 50] hPa")
    print(f"Excluded timestamp : {row.time_utc:%Y-%m-%d %H:%M UTC} -> {', '.join(reasons)}")

print("Reason for blank   : operational UTCI requires 10 m wind in [0.5, 17.0] m/s; sub-0.5 m/s winds are left unclipped rather than floored.")
print()
print(f"{'Day':24s} {'Valid':7s} {'ws10 [m/s]':30s} {'UTCI [C]':27s}")
print("-" * 92)
for date_str, label in SEASONAL_DATES.items():
    sub = df_days[df_days["date_str"] == date_str]
    sub_valid = sub[sub["utci"].notna()]
    print(
        f"{label:24s} "
        f"{len(sub_valid):2d} / {len(sub):2d} "
        f"{_fmt_minmax(sub, 'ws10', 'm/s'):30s} "
        f"{_fmt_minmax(sub_valid, 'utci', 'C'):27s}"
    )

In [ ]:
df_days["utci_class"] = classify(values=df_days["utci"], scale=UTCI_SCALE)

utci_labels = df_days["utci_class"].fillna("Blank (out of range)")
utci_order = [band.label for band in UTCI_SCALE.bands] + ["Blank (out of range)"]
utci_counts = (
    pd.crosstab(utci_labels, df_days["date_str"])
    .reindex(index=utci_order, columns=list(SEASONAL_DATES), fill_value=0)
)
utci_counts.columns = [SEASONAL_DATES[d] for d in utci_counts.columns]
print("Hourly UTCI stress-category counts per day:")
print(utci_counts.to_string())

utci_fig_path = plot_utci_diurnal(
    df=df_days,
    scale=UTCI_SCALE,
    output_path=OUTPUT_DIR / "fig_2.2_utci_diurnal.png",
    season_labels=SEASONAL_DATES,
)

## Section 3: Comparing PET and UTCI

In [ ]:
paired = df_days[df_days["pet"].notna() & df_days["utci"].notna()].copy()
paired["diff_pet_minus_utci"] = paired["pet"] - paired["utci"]

idx_min_diff = paired["diff_pet_minus_utci"].idxmin()
idx_max_diff = paired["diff_pet_minus_utci"].idxmax()
idx_abs_diff = paired["diff_pet_minus_utci"].abs().idxmax()

same_class = int((paired["pet_class"] == paired["utci_class"]).sum())
diff_class = len(paired) - same_class

print(f"Paired valid hours          : {len(paired)} / {len(df_days)}")
print(f"Mean (PET - UTCI)           : {paired['diff_pet_minus_utci'].mean():+.2f} K")
print(
    f"Min  (PET - UTCI)           : {paired.loc[idx_min_diff, 'diff_pet_minus_utci']:+.2f} K "
    f"at {paired.loc[idx_min_diff, 'time_utc']:%Y-%m-%d %H:%M UTC}"
)
print(
    f"Max  (PET - UTCI)           : {paired.loc[idx_max_diff, 'diff_pet_minus_utci']:+.2f} K "
    f"at {paired.loc[idx_max_diff, 'time_utc']:%Y-%m-%d %H:%M UTC}"
)
print(
    f"Max |PET - UTCI|            : {abs(paired.loc[idx_abs_diff, 'diff_pet_minus_utci']):.2f} K "
    f"at {paired.loc[idx_abs_diff, 'time_utc']:%Y-%m-%d %H:%M UTC}"
)
print(f"Stress-class name agreement : {same_class} matching / {diff_class} differing (out of {len(paired)} hours)")

comp_fig_path = plot_pet_utci_comparison(
    df=df_days,
    output_path=OUTPUT_DIR / "fig_3.1_pet_utci_comparison.png",
    season_labels=SEASONAL_DATES,
)

In [ ]:
# Radiative sensitivity evaluates observed Tmrt against neutral radiation (Tmrt = Ta) at the selected UTC timestamp.
TARGET_TIMESTAMP = "2012-07-11 12:00 UTC"

target_utc = pd.Timestamp(TARGET_TIMESTAMP)
if target_utc.tzinfo is None:
    target_utc = target_utc.tz_localize("UTC")
else:
    target_utc = target_utc.tz_convert("UTC")

matched = df_days[df_days["time_utc"] == target_utc]
if matched.empty:
    raise KeyError(f"Timestamp {target_utc:%Y-%m-%d %H:%M UTC} not found in df_days.")
row = matched.iloc[0]

def _eval_single_hour(ta: float, vp: float, tmrt: float, ws1_1: float, ws10: float) -> tuple[float, str, float, str]:
    pet_val = float(biometeo.PET(Ta=ta, VP=vp, Tmrt=tmrt, v=ws1_1)["PET_v"])
    pet_lbl = str(classify(values=np.array([pet_val]), scale=PET_SCALE)[0])
    is_valid = bool(mask_utci_inputs(ta=np.array([ta]), tmrt=np.array([tmrt]), ws10=np.array([ws10]), vp=np.array([vp]))[0])
    if is_valid:
        utci_val = float(biometeo.UTCI(Ta=ta, VP=vp, Tmrt=tmrt, v=ws10))
        utci_lbl = str(classify(values=np.array([utci_val]), scale=UTCI_SCALE)[0])
    else:
        utci_val = float("nan")
        utci_lbl = "Blank (out of range)"
    return pet_val, pet_lbl, utci_val, utci_lbl

pet_obs, pet_obs_lbl, utci_obs, utci_obs_lbl = _eval_single_hour(
    row["ta"], row["vp"], row["tmrt"], row["ws1_1"], row["ws10"]
)
pet_shade, pet_shade_lbl, utci_shade, utci_shade_lbl = _eval_single_hour(
    row["ta"], row["vp"], row["ta"], row["ws1_1"], row["ws10"]
)

print(
    f"Timestamp {target_utc:%Y-%m-%d %H:%M UTC} | "
    f"Ta = {row['ta']:.2f} C, Tmrt = {row['tmrt']:.2f} C, VP = {row['vp']:.2f} hPa, "
    f"ws1_1 = {row['ws1_1']:.2f} m/s, ws10 = {row['ws10']:.2f} m/s"
)
print(f"{'Case':28s} {'PET [C]':>9s} {'PET class':24s} {'UTCI [C]':>9s} {'UTCI class':24s}")
print("-" * 98)
utci_obs_str = f"{utci_obs:9.2f}" if np.isfinite(utci_obs) else "      N/A"
utci_shade_str = f"{utci_shade:9.2f}" if np.isfinite(utci_shade) else "      N/A"
print(f"{'Observed Tmrt (Week 4)':28s} {pet_obs:9.2f} {pet_obs_lbl:24s} {utci_obs_str} {utci_obs_lbl:24s}")
print(f"{'Neutral radiation (Tmrt=Ta)':28s} {pet_shade:9.2f} {pet_shade_lbl:24s} {utci_shade_str} {utci_shade_lbl:24s}")
utci_delta_str = f"{utci_shade - utci_obs:+9.2f}" if (np.isfinite(utci_obs) and np.isfinite(utci_shade)) else "      N/A"
print(f"{'Shift (Neutral - Observed)':28s} {pet_shade - pet_obs:+9.2f} {'':24s} {utci_delta_str}")

## Section 4: Gridded UTCI Over Europe

In [ ]:
# Adapted from ECMWF thermofeel-earthkit-demo (Apache-2.0, (C) ECMWF) and the Week 3 notebook.
# Deferred imports keep Setup and Sections 1-3 runnable without the gridded forecast stack installed.
from datetime import datetime, timedelta

import earthkit.data as ekd
from earthkit.meteo import solar, wind
import thermofeel as tf
from thermofeel.approximations import approximate_fdir_erbs

if FORECAST_STEP_H <= RADIATION_WINDOW_H:
    raise ValueError(
        f"FORECAST_STEP_H ({FORECAST_STEP_H}) must exceed RADIATION_WINDOW_H ({RADIATION_WINDOW_H})."
    )

surface_request = dict(
    type="fc",
    stream="oper",
    levtype="sfc",
    step=FORECAST_STEP_H,
    param=["2t", "2d", "10u", "10v"],
)
if RUN_DATE is None:
    data_source = "ecmwf"
else:
    data_source = "azure"
    surface_request.update(date=RUN_DATE, time=RUN_TIME)

surface_fields = {
    f.metadata("shortName"): f
    for f in ekd.from_source("ecmwf-open-data", source=data_source, request=surface_request).to_fieldlist()
}
t2m_k = surface_fields["2t"].to_numpy()   # [K]
d2m_k = surface_fields["2d"].to_numpy()   # [K]
u10 = surface_fields["10u"].to_numpy()    # [m/s]
v10 = surface_fields["10v"].to_numpy()    # [m/s]
lat2d, lon2d = surface_fields["2t"].geography.latlons()

base_time = datetime.fromisoformat(str(surface_fields["2t"].metadata("base_datetime")))
valid_time = datetime.fromisoformat(str(surface_fields["2t"].metadata("valid_datetime")))

# Differencing FORECAST_STEP_H minus step_prev converts accumulated fields to mean fluxes over RADIATION_WINDOW_H.
step_prev = FORECAST_STEP_H - RADIATION_WINDOW_H
rad_request = dict(
    type="fc",
    stream="oper",
    levtype="sfc",
    param=["ssrd", "ssr", "strd", "str"],
    date=base_time.strftime("%Y%m%d"),
    time=base_time.hour,
    step=[step_prev, FORECAST_STEP_H],
)
rad_fields = {
    (f.metadata("shortName"), int(f.metadata("step"))): f.to_numpy()
    for f in ekd.from_source("ecmwf-open-data", source=data_source, request=rad_request).to_fieldlist()
}
window_seconds = RADIATION_WINDOW_H * 3600.0
ssrd_mean = np.maximum((rad_fields["ssrd", FORECAST_STEP_H] - rad_fields["ssrd", step_prev]) / window_seconds, 0.0)
ssr_mean = np.maximum((rad_fields["ssr", FORECAST_STEP_H] - rad_fields["ssr", step_prev]) / window_seconds, 0.0)
strd_mean = np.maximum((rad_fields["strd", FORECAST_STEP_H] - rad_fields["strd", step_prev]) / window_seconds, 0.0)
# Net thermal radiation (str) is signed and typically negative when upward exceeds downward, so do not clamp at zero.
str_mean = (rad_fields["str", FORECAST_STEP_H] - rad_fields["str", step_prev]) / window_seconds

south_box, north_box, west_box, east_box = EUROPE_BOX
europe_box = (lat2d >= south_box) & (lat2d <= north_box) & (lon2d >= west_box) & (lon2d <= east_box)

window_start = valid_time - timedelta(hours=RADIATION_WINDOW_H)
cossza = solar.cos_solar_zenith_angle_integrated(
    window_start, valid_time, lat2d[europe_box], lon2d[europe_box]
)
day_of_year = valid_time.timetuple().tm_yday

fdir_flux = approximate_fdir_erbs(ssrd_mean[europe_box], cossza, doy=day_of_year)  # [W/m2]
dsrp_flux = tf.approximate_dsrp(fdir=fdir_flux, cossza=cossza)                      # [W/m2]
mrt_box_k = tf.calculate_mean_radiant_temperature(
    ssrd=ssrd_mean[europe_box],
    ssr=ssr_mean[europe_box],
    dsrp=dsrp_flux,
    strd=strd_mean[europe_box],
    fdir=fdir_flux,
    strr=str_mean[europe_box],
    cossza=cossza,
)

ta_box_c = tf.kelvin_to_celsius(t2m_k[europe_box])
tmrt_box_c = tf.kelvin_to_celsius(mrt_box_k)
ws10_box = wind.speed(u10[europe_box], v10[europe_box])
vp_box_hpa = tf.calculate_saturation_vapour_pressure(d2m_k[europe_box])

valid_box = mask_utci_inputs(ta=ta_box_c, tmrt=tmrt_box_c, ws10=ws10_box, vp=vp_box_hpa)
utci_box_c = np.full(ta_box_c.shape, np.nan, dtype=float)
if valid_box.any():
    utci_box_k = tf.calculate_utci(
        t2_k=t2m_k[europe_box][valid_box],
        va=ws10_box[valid_box],
        mrt=mrt_box_k[valid_box],
        ehPa=vp_box_hpa[valid_box],
    )
    utci_box_c[valid_box] = tf.kelvin_to_celsius(utci_box_k)

utci_europe_c = np.full(t2m_k.shape, np.nan, dtype=float)
utci_europe_c[europe_box] = utci_box_c
utci_valid_grid = np.zeros(t2m_k.shape, dtype=bool)
utci_valid_grid[europe_box] = valid_box

if STATS_REGION not in STATS_REGIONS:
    raise KeyError(f"Unknown STATS_REGION '{STATS_REGION}'; expected one of {list(STATS_REGIONS)}")
south_reg, north_reg, west_reg, east_reg = STATS_REGIONS[STATS_REGION]
region_mask = (lat2d >= south_reg) & (lat2d <= north_reg) & (lon2d >= west_reg) & (lon2d <= east_reg)
region_utci = utci_europe_c[region_mask]
region_total = int(region_mask.sum())
region_valid = int(utci_valid_grid[region_mask].sum())
region_masked = region_total - region_valid

print(
    f"ECMWF open-data run ({data_source}): base {base_time:%Y-%m-%d %H:%M UTC}, "
    f"valid {valid_time:%Y-%m-%d %H:%M UTC} (+{FORECAST_STEP_H} h)"
)
print(
    "Note: Direct solar radiation (fdir) is absent from ECMWF open data and is estimated "
    "via the estimation-grade Erbs et al. (1982) method from 3 h mean fluxes (steps 9-12)."
)
print(
    f"Europe box ({south_box} to {north_box} N, {west_box} to {east_box} E): "
    f"{int(valid_box.sum())} / {int(europe_box.sum())} valid cells ({int((~valid_box).sum())} masked)"
)
if np.isfinite(region_utci).any():
    print(
        f"{STATS_REGION} ({south_reg} to {north_reg} N, {west_reg} to {east_reg} E): "
        f"min = {np.nanmin(region_utci):.2f} C, mean = {np.nanmean(region_utci):.2f} C, "
        f"max = {np.nanmax(region_utci):.2f} C | "
        f"valid = {region_valid} / {region_total}, masked = {region_masked}"
    )
else:
    print(f"{STATS_REGION}: n/a (all {region_total} grid cells masked outside UTCI validity bounds)")

In [ ]:
europe_fig_path = plot_europe_utci_map(
    utci_c=utci_europe_c,
    lat2d=lat2d,
    lon2d=lon2d,
    scale=UTCI_SCALE,
    output_path=OUTPUT_DIR / "fig_4.2_europe_utci.png",
    valid_label=valid_time.strftime("%Y-%m-%d %H:%M UTC"),
)

---

**🦉 Crafted with wisdom at One Weather Lab (OWL)**<br>
Laboratory of Meteorology and Climatology, Physics Department, University of Ioannina<br>
Christos Giannaros <<chris.giannaros@uoi.gr>>